In [1]:
from enum import unique
import joblib
from wildlife_datasets.datasets import Lynx
from wildlife_tools.data import WildlifeDataset
import torchvision.transforms as T
import torch
import timm
import torchvision.models as models
from wildlife_tools.features import DeepFeatures
from wildlife_tools.similarity import CosineSimilarity
from wildlife_tools.inference import KnnClassifier
import numpy as np
from itertools import product
from proportional_split import proportional_split


c:\ob\BP\pythonProject1\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
transform224 = T.Compose([
    # T.CenterCrop(256),
    T.Resize([224,224]),
    # transform to grayscale here
    # T.Grayscale(num_output_channels=3),
    T.ToTensor(),
    T.Normalize(
        mean=(0.485, 0.456, 0.406),
        std=(0.229, 0.224, 0.225)
    )
])
transform384 = T.Compose([
    # T.CenterCrop(256),
    T.Resize([384,384]),
    # transform to grayscale here
    # T.Grayscale(num_output_channels=3),
    T.ToTensor(),
    T.Normalize(
        mean=(0.485, 0.456, 0.406),
        std=(0.229, 0.224, 0.225)
    )
])

In [4]:
# from collections import Counter
# Counter(metadata_transformed.df['identity'])
metadata_transformed = Lynx('data_rysy/rys_trening_data_Beno')

In [5]:
def create_WildlifeDatasets(dataset_database_df, dataset_query_df, transform):
    return (WildlifeDataset(
        dataset_database_df,
        metadata_transformed.root,
        transform=transform
    ),
   WildlifeDataset(
        dataset_query_df,
        metadata_transformed.root,
        transform=transform
    ))


def evaluate_accuracy(dataset_database, dataset_query, extractor, query_ratio):
    # Feature extraction
    query = extractor(dataset_query)
    database = extractor(dataset_database)
    
    # Similarity computation
    similarity_function = CosineSimilarity()
    similarity = similarity_function(query, database)
    
    # Classification and accuracy
    classifier = KnnClassifier(k=1, database_labels=dataset_database.labels_string)
    predictions = classifier(similarity['cosine'])
    accuracy = np.mean(dataset_query.labels_string == predictions)
    print(f"Query ratio: {query_ratio}  Accuracy: {accuracy}")
    return accuracy


def get_similarity(dataset_database, dataset_query, extractor):
    query = extractor(dataset_query)
    database = extractor(dataset_database)
    similarity_function = CosineSimilarity()
    return similarity_function(query, database)

In [6]:
# Device configuration
if __name__ == '__main__':
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(device)
    # exit()
    dataset_versions = ['', '_detected', '_detected_manual']
    # descriptor_versions = ['-T-224', '-S-224', '-B-224', '-L-384']
    descriptor_versions = [ '-T-224']

    for dataset_version, descriptor_version in product(dataset_versions, descriptor_versions):

        # Load metadata pointing to transformed images
        print(f'data_rysy/rys_trening_data_Beno{dataset_version}')
        metadata_transformed = Lynx(f'data_rysy/rys_trening_data_Beno{dataset_version}')

        # Define transform with ToTensor and Normalize only
        # multicrop_transform = make_multicrop_transform(5, 'mean', 224)
        transform = transform224 if descriptor_version.endswith('-224') else transform384

        dataset_database_df, dataset_query_df = proportional_split(metadata_transformed.df, query_ratio=0.2)
        dataset_database, dataset_query = create_WildlifeDatasets(dataset_database_df, dataset_query_df, transform)

        name = 'hf-hub:BVRA/MegaDescriptor' + descriptor_version
        extractor = DeepFeatures(timm.create_model(name, num_classes=0, pretrained=True), batch_size=16, device=str(device), num_workers=0, pool='max')

        # evaluate_accuracy(dataset_database, dataset_query, extractor, 0.2)
        query = extractor(dataset_query)       # shape (N_query, K)
        database = extractor(dataset_database) # shape (N_db, K)

        # If `query` and `database` are numpy arrays, convert them:
        query_tensor = torch.from_numpy(query)
        database_tensor = torch.from_numpy(database)

        # Combine into one tensor
        all_embeddings = torch.cat([query_tensor, database_tensor], dim=0)

        # Save to emb.pt
        descriptor_abb = descriptor_version[-5:]
        torch.save(all_embeddings, f"saved_models/{descriptor_abb}/embeddings/emb{dataset_version}.pt")

        # Suppose this column contains class names
        labels = metadata_transformed.df["identity"].values

        # Encode text labels -> integers
        from sklearn.preprocessing import LabelEncoder
        encoder = LabelEncoder()
        label_ids = encoder.fit_transform(labels)

        # Save both        
        torch.save(label_ids, f"saved_models/{descriptor_abb}/labels/labels{dataset_version}.pt")
        joblib.dump(encoder, f"saved_models/{descriptor_abb}/label_encoders/label_encoder{dataset_version}.pkl")
        print(f"Saved embeddings and labels for dataset version {dataset_version} and descriptor {descriptor_version}.")
        
    
    

cpu
data_rysy/rys_trening_data_Beno


INFO:timm.models._builder:Loading pretrained weights from Hugging Face hub (BVRA/MegaDescriptor-T-224)


TypeError: DeepFeatures.__init__() got an unexpected keyword argument 'pool'

In [5]:
from torchvision import transforms

class MultiCropTransform:
    def __init__(self, num_crops=5, pool='max', image_size=224):
        self.num_crops = num_crops
        self.pool = pool
        self.image_size = image_size
        
        # Update the transforms to use the dynamic image_size
        self.base_transform = transforms.Compose([
            transforms.Resize(int(image_size * 1.5)),  # Resize to 1.5 * image_size to ensure enough pixels
            transforms.RandomResizedCrop(self.image_size, scale=(0.5, 1.0)),  # Use image_size dynamically
            transforms.RandomHorizontalFlip(),
            transforms.ToTensor(),
            transforms.Normalize(
                mean=(0.485, 0.456, 0.406),
                std=(0.229, 0.224, 0.225)
            ),
        ])

    def __call__(self, image):
        crops = [self.base_transform(image) for _ in range(self.num_crops)]  # [N, 3, H, W]
        if self.pool == 'mean':
            return torch.stack(crops).mean(dim=0)
        elif self.pool == 'max':
            return torch.stack(crops).max(dim=0).values
        elif self.pool is None:
            return crops
        else:
            return torch.stack(crops)  # raw crops, if your model handles them



def make_multicrop_transform(num_crops=5, pool='max', image_size=224):
    return MultiCropTransform(num_crops=num_crops, pool=pool)



In [9]:
name = 'hf-hub:BVRA/MegaDescriptor-T-224' 
extractor = DeepFeatures(timm.create_model(name, num_classes=0, pretrained=True), batch_size=16, device=str(device), num_workers=0)
evaluate_accuracy(dataset_database, dataset_query, extractor, 0.2)

INFO:timm.models._builder:Loading pretrained weights from Hugging Face hub (BVRA/MegaDescriptor-T-224)
100%|███████████████████████████████████████████████████████████████| 16/16 [01:23<00:00,  5.20s/it]

Query ratio: 0.2  Accuracy: 0.6714285714285714



c:\ob\BP\pythonProject1\.venv\lib\site-packages\wildlife_tools\inference\classifier.py:73: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  results = pd.DataFrame(results).T.fillna(method="ffill").T


0.6714285714285714

In [28]:
type(dataset_database)

wildlife_tools.data.dataset.WildlifeDataset

In [27]:
type(dataset)

wildlife_tools.data.dataset.WildlifeDataset

In [26]:
dataset = WildlifeDataset(metadata_transformed.df, transform=transform224)

embeddings = extractor(dataset_database)  # numpy array (N, D)
embeddings = torch.from_numpy(embeddings)


  0%|                                                                        | 0/16 [00:02<?, ?it/s]


KeyboardInterrupt: 

In [ ]:
from sklearn.manifold import TSNE

tsne = TSNE(n_components=2, perplexity=30)
emb_2d = tsne.fit_transform(embeddings)

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm

unique_labels = np.unique(labels)
num_classes = len(unique_labels)

# Use high-contrast categorical colormap
cmap = cm.get_cmap('tab20', num_classes)

plt.figure(figsize=(8, 8))

for i, label in enumerate(unique_labels):
    mask = labels == label
    plt.scatter(
        emb_2d[mask, 0],
        emb_2d[mask, 1],
        s=12,
        color=cmap(i),
        label=str(label)
    )

plt.title("t-SNE visualization of embeddings")
plt.xlabel("Dimension 1")
plt.ylabel("Dimension 2")

plt.legend(
    bbox_to_anchor=(1.05, 1),
    loc='upper left',
    borderaxespad=0.,
    fontsize=8
)

plt.tight_layout()
plt.show()

NameError: name 'all_embeddings' is not defined

In [ ]:
extractor.model.state_dict()

In [ ]:
# Testovanie splitov
# (tie nie zaujimave)
accuracies = []
for query_ratio in np.arange(0.2, 0.8, 0.05):
    dataset_database_df, dataset_query_df = proportional_split(metadata_transformed.df, query_ratio=query_ratio)
    dataset_database, dataset_query = create_WildlifeDatasets(dataset_database_df, dataset_query_df, transform)
    accuracies.append(evaluate_accuracy(dataset_database, dataset_query, extractor, query_ratio))

In [14]:
from sklearn.model_selection import KFold

def stratified_k_fold_split(df, query_ratio=0.2, n_splits=5):
    lynx_groups = df.groupby("identity")  # Group by lynx
    
    # Prepare storage for the splits
    folds = [[] for _ in range(n_splits)]
    
    # For each lynx, assign its images to different folds
    for lynx in df['identity'].unique():
        lynx_images = lynx_groups.get_group(lynx).sort_values("path")  # Sort images by path
        indices = lynx_images.index.to_numpy()
        num_test = max(1, int(len(indices) * query_ratio))  # Ensure at least one test sample
        
        # Split into k folds
        kf = KFold(n_splits=n_splits, shuffle=False)
        for fold_idx, (train_idx, val_idx) in enumerate(kf.split(indices)):
            folds[fold_idx].append((indices[train_idx], indices[val_idx]))
    
    # Convert folds to proper format
    final_folds = []
    for fold in folds:
        train_idx = np.concatenate([x[0] for x in fold])
        test_idx = np.concatenate([x[1] for x in fold])
        final_folds.append((train_idx, test_idx))
    
    return final_folds


<h2>Cross Validacia</h2>

In [ ]:
cross_val_splits = stratified_k_fold_split(metadata_transformed.df, n_splits=3)

for d,q in cross_val_splits:
    print("\n\nTRAIN")
    print(np.array(metadata_transformed.df.values[:, 2])[d])
    print("\n\nTEST")
    print(np.array(metadata_transformed.df.values[:, 2])[q])

accuracies_5folds = []
for d_indices, q_indices in cross_val_splits:
    dataset_database, dataset_query = create_WildlifeDatasets(metadata_transformed.df.iloc[d_indices], metadata_transformed.df.iloc[q_indices], transform)
    accuracies_5folds.append(evaluate_accuracy(dataset_database, dataset_query, extractor, 0.2))

<h2>Zmensujuca sa DB</h2>

In [16]:
def progressive_query_split_ratio(df, query_ratio=0.33, n_folds=5):
    groups = df.groupby("identity")
    folds = []

    for fold_idx in range(n_folds):
        train_indices = []
        test_indices = []

        for identity, group in groups:
            group = group.sort_values("path")
            indices = group.index.to_numpy()

            num_query = max(1, int(len(indices) * query_ratio))
            query_idx = indices[-num_query:]
            db_candidates = indices[:-num_query]
            num_db_total = len(db_candidates)

            if num_db_total == 0:
                db_idx = []
            else:
                # Progressive ratio (start at 1/n_folds instead of 0)
                ratio = (fold_idx + 1) / n_folds
                num_db_this_fold = int(round(num_db_total * ratio))

                # Ensure at least 1 DB sample if possible
                if num_db_this_fold == 0 and num_db_total > 0:
                    num_db_this_fold = 1

                db_idx = db_candidates[:num_db_this_fold]

            train_indices.extend(db_idx)
            test_indices.extend(query_idx)

        folds.append((np.array(train_indices), np.array(test_indices)))

    return folds




In [ ]:
progressive_folds = progressive_query_split_ratio(metadata_transformed.df, query_ratio=0.2, n_folds=10)
from collections import Counter

for i, (train_idx, test_idx) in enumerate(progressive_folds):
    print(f"Fold {i}:")
    print("D:", Counter(metadata_transformed.df['identity'][train_idx]))
    print("Q:", Counter(metadata_transformed.df['identity'][test_idx]))
    print("---")


In [18]:
Counter(metadata_transformed.df['identity'])

Counter({'Benadik': 57,
         'Izidor': 34,
         'Milos': 33,
         'Albin': 30,
         'Edo': 24,
         'Roman': 22,
         'Kiara': 13,
         'Dio': 8,
         'Eliska': 8,
         'Lubos': 8,
         'Zora': 8,
         'Brano': 7,
         'Adam': 5,
         'Silvester': 3})

In [ ]:
accuracies_10progressive_folds = []
for d_indices, q_indices in progressive_folds:
    dataset_database, dataset_query = create_WildlifeDatasets(metadata_transformed.df.iloc[d_indices], metadata_transformed.df.iloc[q_indices], transform)
    accuracies_10progressive_folds.append(evaluate_accuracy(dataset_database, dataset_query, extractor, 0.2))

In [20]:
accuracies_10progressive_folds

[0.34782608695652173,
 0.41304347826086957,
 0.45652173913043476,
 0.5217391304347826,
 0.5434782608695652,
 0.5652173913043478,
 0.5652173913043478,
 0.5217391304347826,
 0.5217391304347826,
 0.5652173913043478]

<h2>Test disjunktnosti splitov</h2>

In [21]:
# disjoint database and query sets test
for fold_idx, (d, q) in enumerate(progressive_folds):
    train_set = set(metadata_transformed.df.values[d, 2])  # Convert train paths to a set
    test_set = set(metadata_transformed.df.values[q, 2])   # Convert test paths to a set

    intersection = train_set & test_set  # Compute the intersection
    
    print(f"Fold {fold_idx + 1}:")
    print(f"Train Size: {len(train_set)}, Test Size: {len(test_set)}, Intersection Size: {len(intersection)}")
    
    # Assert that there is no intersection
    assert len(intersection) == 0, f"Error: Train and Test sets are not disjoint in Fold {fold_idx + 1}!"
    
    print("✅ Train and Test sets are disjoint!\n")


Fold 1:
Train Size: 25, Test Size: 46, Intersection Size: 0
✅ Train and Test sets are disjoint!

Fold 2:
Train Size: 42, Test Size: 46, Intersection Size: 0
✅ Train and Test sets are disjoint!

Fold 3:
Train Size: 63, Test Size: 46, Intersection Size: 0
✅ Train and Test sets are disjoint!

Fold 4:
Train Size: 86, Test Size: 46, Intersection Size: 0
✅ Train and Test sets are disjoint!

Fold 5:
Train Size: 110, Test Size: 46, Intersection Size: 0
✅ Train and Test sets are disjoint!

Fold 6:
Train Size: 128, Test Size: 46, Intersection Size: 0
✅ Train and Test sets are disjoint!

Fold 7:
Train Size: 151, Test Size: 46, Intersection Size: 0
✅ Train and Test sets are disjoint!

Fold 8:
Train Size: 173, Test Size: 46, Intersection Size: 0
✅ Train and Test sets are disjoint!

Fold 9:
Train Size: 191, Test Size: 46, Intersection Size: 0
✅ Train and Test sets are disjoint!

Fold 10:
Train Size: 214, Test Size: 46, Intersection Size: 0
✅ Train and Test sets are disjoint!



In [22]:
accuracies_5folds

[0.5054945054945055, 0.5057471264367817, 0.45121951219512196]

In [23]:
def simple_path(path):
    return path.split("\\")[-1]

In [24]:
image_classes = np.array(metadata_transformed.df.values[:, 1])  # Shape: (321, 4)

# Get cosine similarity matrix
similarity = get_similarity(dataset_database, dataset_query, extractor)
cosine_sim = similarity["cosine"]  # Shape: (n_predictions, 321)

# Get top 5 highest values for each row
top5_indices = np.argsort(-cosine_sim, axis=1)[:, :5]  # Sort in descending order, take top 5 indices

# Retrieve similarity scores for those indices
top5_scores = np.take_along_axis(cosine_sim, top5_indices, axis=1)
top5_classes = image_classes[top5_indices]
for i, (classes, scores) in enumerate(zip(top5_classes, top5_scores)):
    print(f"Image {simple_path(dataset_query_df.iloc[i]['path'])}:")
    for j, (cls, score) in enumerate(zip(classes, scores)):
        print(f"  {simple_path(metadata_transformed.df.values[top5_indices[i][j], 2])}: {score:.4f}")
    print()

# Map indices to actual class names using image_classes
top5_classes = image_classes[top5_indices]




100%|███████████████████████████████████████████████████████████████| 14/14 [00:03<00:00,  4.53it/s]

Image Adam_2.JPG:
  Adam_3.JPG: 0.6718
  Adam_2.JPG: 0.6177
  Izidor_20.JPG: 0.5627
  Adam_4.JPG: 0.5551
  Izidor_6.JPG: 0.4972

Image Adam_5.JPG:
  Izidor_5.JPG: 0.5035
  Benadik_50.JPG: 0.4955
  Edo_14.JPG: 0.4689
  Eliska_6.JPG: 0.3633
  Milos_35.JPG: 0.3628

Image Adam_3.JPG:
  Izidor_12.JPG: 0.6179
  Milos_35.JPG: 0.4841
  Milos_31.JPG: 0.4730
  Izidor_5.JPG: 0.4342
  Benadik_42.JPG: 0.4282

Image Adam_1.JPG:
  Izidor_35.JPG: 0.3097
  Lubos_7.JPG: 0.3087
  Izidor_8.JPG: 0.3064
  Dio_2.JPG: 0.2770
  Izidor_10.JPG: 0.2745

Image Albin_39.jpg:
  Edo_16.JPG: 0.4063
  Kiara_8.JPG: 0.3548
  Milos_12.JPG: 0.3293
  Benadik_14.JPG: 0.3291
  Dio_7.JPG: 0.3222

Image Albin_27.JPG:
  Edo_16.JPG: 0.5811
  Albin_35.JPG: 0.5780
  Albin_23.JPG: 0.5684
  Albin_36.JPG: 0.5420
  Izidor_2.JPG: 0.5298

Image Albin_35.JPG:
  Benadik_65.JPG: 0.5278
  Lubos_2.jpg: 0.5072
  Milos_26.JPG: 0.5051
  Milos_24.JPG: 0.5051
  Izidor_31.JPG: 0.5038

Image Albin_29.JPG:
  Benadik_25.JPG: 0.5500
  Benadik_28.JPG: 0